# Data Collection

This notebook combines camera capture and rover control so you can drive manually and save labeled frames from one panel.

Recommended workflow:

- open the serial port
- open the camera
- drive with the motion buttons
- grab a frame when the rover is in a useful position
- save the frame with the matching steering label


In [ ]:
from pathlib import Path
import csv
import glob
import io
import json
import os
import sys
import time

import cv2
import ipywidgets as widgets
from IPython.display import display
from PIL import Image
import serial
import serial.tools.list_ports

project_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from jetcar.camera import open_camera, read_rgb_frame

video_devices = sorted(glob.glob('/dev/video*'))
print('Project root:', project_root)
print('Video devices:', video_devices)
print('Basic notebook imports ok')


In [ ]:
session_name = time.strftime('%Y%m%d-%H%M%S')
image_dir = project_root / 'data' / 'raw' / session_name
image_dir.mkdir(parents=True, exist_ok=True)
csv_path = image_dir / 'labels.csv'
fieldnames = ['image_path', 'steering']

if not csv_path.exists():
    with csv_path.open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

print('Saving images to:', image_dir)
print('Saving labels to:', csv_path)


In [ ]:
camera_handle = None
serial_handle = None
current_frame = None
frame_index = 0
last_drive_steering = 0.0


In [ ]:
def available_ports():
    ports = [port.device for port in serial.tools.list_ports.comports()]
    for fallback in ('/dev/ttyTHS1', '/dev/ttyTHS2'):
        if os.path.exists(fallback) and fallback not in ports:
            ports.append(fallback)
    return ports


def next_frame_name():
    global frame_index
    name = f'frame_{frame_index:05d}.jpg'
    frame_index += 1
    return name


def render_frame(frame_rgb):
    image = Image.fromarray(frame_rgb)
    with io.BytesIO() as buf:
        image.save(buf, format='JPEG')
        preview.value = buf.getvalue()


def set_status(text):
    status.value = f'<b>Status:</b> {text}'


def log(message):
    with log_output:
        print(message)


def speed_cmd(left, right):
    return {'T': 1, 'L': float(left), 'R': float(right)}


def open_serial_device(port, baudrate, timeout=0.4):
    ser = serial.Serial(port, baudrate=baudrate, timeout=timeout)
    ser.reset_input_buffer()
    ser.reset_output_buffer()
    return ser


def close_serial_device():
    global serial_handle
    if serial_handle is not None and serial_handle.is_open:
        serial_handle.close()
    serial_handle = None


def send_json(payload):
    if serial_handle is None or not serial_handle.is_open:
        raise RuntimeError('Serial port is not open.')
    line = json.dumps(payload, separators=(',', ':')) + '\n'
    serial_handle.write(line.encode('utf-8'))
    serial_handle.flush()
    time.sleep(0.12)
    waiting = serial_handle.in_waiting
    response = ''
    if waiting:
        response = serial_handle.read(waiting).decode('utf-8', errors='replace')
    return line.strip(), response


def open_selected_camera():
    global camera_handle
    close_camera()
    camera_handle = open_camera(
        source=camera_source_dropdown.value,
        sensor_id=sensor_id.value,
        device_index=device_index.value,
        width=frame_width.value,
        height=frame_height.value,
        warmup_frames=warmup_frames.value,
    )
    set_status(f'camera open ({camera_source_dropdown.value})')


def close_camera():
    global camera_handle
    if camera_handle is not None:
        camera_handle.release()
    camera_handle = None


def grab_frame():
    global current_frame
    if camera_handle is None:
        raise RuntimeError('Camera is not open.')
    best_frame = None
    best_brightness = -1.0
    for _ in range(6):
        frame = read_rgb_frame(camera_handle)
        brightness = float(frame.mean())
        if brightness > best_brightness:
            best_frame = frame
            best_brightness = brightness
        time.sleep(0.03)
    current_frame = best_frame
    render_frame(current_frame)
    set_status(f'frame captured (mean={best_brightness:.1f})')


def save_current_frame(steering):
    if current_frame is None:
        raise RuntimeError('Capture a frame before saving.')
    image_name = next_frame_name()
    image_path = image_dir / image_name
    frame_bgr = cv2.cvtColor(current_frame, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(image_path), frame_bgr)
    with csv_path.open('a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writerow({'image_path': image_name, 'steering': steering})
    set_status(f'saved {image_name} with steering={steering:.2f}')
    log(f'Saved {image_name} steering={steering:.2f}')


def drive(left, right, steering_label, duration=None):
    global last_drive_steering
    sent, response = send_json(speed_cmd(left, right))
    last_drive_steering = steering_label
    steering_slider.value = steering_label
    log(f'DRIVE {sent}')
    if response:
        log(f'RECV {response}')
    if duration is not None:
        time.sleep(duration)
        send_json(speed_cmd(0.0, 0.0))


def stop_now():
    sent, response = send_json(speed_cmd(0.0, 0.0))
    log(f'STOP {sent}')
    if response:
        log(f'RECV {response}')


def on_click_wrap(fn):
    def wrapped(_):
        try:
            fn()
        except Exception as exc:
            set_status(f'error: {exc}')
            log(f'ERROR: {exc}')
    return wrapped


In [ ]:
port_options = available_ports() or ['/dev/ttyTHS1', '/dev/ttyTHS2']
default_port = '/dev/ttyTHS1' if '/dev/ttyTHS1' in port_options else port_options[0]

device_hint = widgets.HTML(value=f"<b>Video devices:</b> {' '.join(video_devices) if video_devices else 'none found'}")
serial_hint = widgets.HTML(value=f"<b>Serial ports:</b> {' '.join(port_options) if port_options else 'none found'}")

camera_source_dropdown = widgets.Dropdown(options=['usb', 'csi'], value='usb', description='Source')
sensor_id = widgets.IntText(value=0, description='Sensor')
device_index = widgets.IntText(value=0, description='Device')
frame_width = widgets.IntText(value=1280, description='Width')
frame_height = widgets.IntText(value=720, description='Height')
warmup_frames = widgets.IntText(value=12, description='Warmup')

port_dropdown = widgets.Dropdown(options=port_options, value=default_port, description='Port')
baud_input = widgets.IntText(value=115200, description='Baud')
drive_speed = widgets.FloatSlider(value=0.25, min=0.05, max=0.50, step=0.05, description='Drive')
turn_speed = widgets.FloatSlider(value=0.50, min=0.05, max=1.00, step=0.05, description='Turn')
pulse_time = widgets.FloatSlider(value=0.35, min=0.10, max=1.00, step=0.05, description='Pulse s')
steering_slider = widgets.FloatSlider(value=0.0, min=-1.0, max=1.0, step=0.05, description='Steering')

open_port_button = widgets.Button(description='Open Port', button_style='success')
close_port_button = widgets.Button(description='Close Port')
stop_button = widgets.Button(description='Stop', button_style='danger')
forward_button = widgets.Button(description='Forward')
backward_button = widgets.Button(description='Backward')
left_button = widgets.Button(description='Left')
right_button = widgets.Button(description='Right')

open_camera_button = widgets.Button(description='Open Camera', button_style='success')
close_camera_button = widgets.Button(description='Close Camera')
grab_button = widgets.Button(description='Grab Frame', button_style='info')
save_button = widgets.Button(description='Save Current', button_style='warning')
save_last_drive_button = widgets.Button(description='Save Last Drive')
save_left_button = widgets.Button(description='Save Left')
save_straight_button = widgets.Button(description='Save Straight')
save_right_button = widgets.Button(description='Save Right')

status = widgets.HTML(value='<b>Status:</b> idle')
preview = widgets.Image(format='jpeg', width=800, height=450)
log_output = widgets.Output(layout={'border': '1px solid #444', 'max_height': '220px', 'overflow_y': 'auto'})

open_port_button.on_click(on_click_wrap(lambda: [close_serial_device(), globals().__setitem__('serial_handle', open_serial_device(port_dropdown.value, baud_input.value)), set_status(f'serial open on {port_dropdown.value}')]))
close_port_button.on_click(on_click_wrap(lambda: [close_serial_device(), set_status('serial closed')]))
stop_button.on_click(on_click_wrap(lambda: [stop_now(), set_status('stop sent')]))
forward_button.on_click(on_click_wrap(lambda: [drive(drive_speed.value, drive_speed.value, 0.0, duration=pulse_time.value), set_status('forward pulse sent')]))
backward_button.on_click(on_click_wrap(lambda: [drive(-drive_speed.value, -drive_speed.value, 0.0, duration=pulse_time.value), set_status('backward pulse sent')]))
left_button.on_click(on_click_wrap(lambda: [drive(-turn_speed.value, turn_speed.value, -0.6, duration=pulse_time.value), set_status('left pulse sent')]))
right_button.on_click(on_click_wrap(lambda: [drive(turn_speed.value, -turn_speed.value, 0.6, duration=pulse_time.value), set_status('right pulse sent')]))

open_camera_button.on_click(on_click_wrap(lambda: open_selected_camera()))
close_camera_button.on_click(on_click_wrap(lambda: [close_camera(), set_status('camera closed')]))
grab_button.on_click(on_click_wrap(lambda: grab_frame()))
save_button.on_click(on_click_wrap(lambda: save_current_frame(steering_slider.value)))
save_last_drive_button.on_click(on_click_wrap(lambda: save_current_frame(last_drive_steering)))
save_left_button.on_click(on_click_wrap(lambda: save_current_frame(-0.6)))
save_straight_button.on_click(on_click_wrap(lambda: save_current_frame(0.0)))
save_right_button.on_click(on_click_wrap(lambda: save_current_frame(0.6)))

panel = widgets.VBox([
    widgets.HTML(value='<h3>Camera</h3>'),
    device_hint,
    widgets.HBox([camera_source_dropdown, sensor_id, device_index]),
    widgets.HBox([frame_width, frame_height, warmup_frames]),
    widgets.HBox([open_camera_button, close_camera_button, grab_button]),
    widgets.HTML(value='<h3>Drive</h3>'),
    serial_hint,
    widgets.HBox([port_dropdown, baud_input]),
    drive_speed,
    turn_speed,
    pulse_time,
    widgets.HBox([open_port_button, close_port_button, stop_button]),
    widgets.HBox([forward_button, backward_button, left_button, right_button]),
    widgets.HTML(value='<h3>Label And Save</h3>'),
    steering_slider,
    widgets.HBox([save_button, save_last_drive_button, save_left_button, save_straight_button, save_right_button]),
    status,
    preview,
    log_output,
])

display(panel)


## Notes

- Use `Save Last Drive` right after a motion button to save the same steering label you just used.
- For the first dataset, a few hundred good examples is enough to get started.
- Balance straight, left, right, and recovery frames so the model does not learn to only go straight.
